# The Rise of LLMs: From Tradition to Attention
## Week 7 — Day 1 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Understand what Large Language Models are and what they are designed for
- Learn about Transformer architecture and tokenization
- Differentiate Pretraining from Fine-Tuning
- Generate text using a pretrained LLM (GPT-2)

> **Run on Google Colab** — GPU optional for GPT-2 inference.

## 🌟 Exercise 1 — What are Large Language Models (LLMs)?

### 1. Explanation of LLMs

**Large Language Models (LLMs)** are deep learning models trained on massive amounts of text data to understand and generate human language. They are built on the **Transformer architecture**, which uses a mechanism called *self-attention* to weigh the importance of every word in a sentence relative to every other word — allowing the model to capture long-range dependencies in text.

LLMs are designed for a wide range of Natural Language Processing (NLP) tasks:
- **Text generation** — writing coherent paragraphs, stories, code
- **Question answering** — reading a passage and answering questions about it
- **Summarization** — condensing long documents into short summaries
- **Translation** — converting text from one language to another
- **Sentiment analysis** — classifying the emotional tone of a text
- **Conversational AI** — powering chatbots and virtual assistants

What makes them *large* is the scale: modern LLMs like GPT-4 or Llama-3 have **billions of parameters** and are trained on hundreds of billions of tokens scraped from books, websites, and code repositories. This scale gives them emergent abilities — capabilities that appear only at a certain model size, like multi-step reasoning or in-context learning from a few examples.

### 2. Install & Import Libraries

In [ ]:
# Install necessary libraries
!pip install transformers matplotlib --quiet

# Import required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported ✓")

### 3. Load a Pretrained GPT-2 Model and Tokenizer

In [ ]:
# Loading a pretrained model and tokenizer
model_name = "gpt2"  # GPT-2 is used here for demonstration; can be replaced with models like "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print(f"  Parameters : {model.num_parameters():,}")
print(f"  Vocab size : {tokenizer.vocab_size:,}")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence. 
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")

## 🌟 Exercise 2 — Transformer Architecture and Tokenization

### Explanation of Tokenization

**Tokenization** is the process of converting raw text into a sequence of smaller units called **tokens**, which the model can process numerically. A token can be a whole word, a sub-word, a character, or even a punctuation mark depending on the tokenization strategy used.

GPT-2 uses **Byte-Pair Encoding (BPE)** tokenization:
1. The text is first split into characters.
2. The most frequent pairs of adjacent units are merged iteratively until a vocabulary of a fixed size (50,257 for GPT-2) is reached.
3. Common words become single tokens (e.g. `"the"` → one token), while rare words are split into sub-word pieces (e.g. `"tokenization"` → `["token", "ization"]`).

This allows the model to handle any word — even ones never seen during training — by decomposing them into known sub-units, providing a good balance between vocabulary size and coverage.

In [ ]:
text = "Artificial intelligence is transforming the world one model at a time."

# 2. Tokenize input text
tokens    = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text : {text}")
print(f"\nTokens    ({len(tokens)})  : {tokens}")
print(f"Token IDs ({len(token_ids)}) : {token_ids}")

# 3. Visualize the tokenization process
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(tokens)), token_ids, color='skyblue', edgecolor='white')
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
ax.set_xlabel("Tokens")
ax.set_ylabel("Token IDs")
ax.set_title(f'GPT-2 Tokenization — "{text[:40]}..."', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Annotate bars with ID values
for bar, tid in zip(bars, token_ids):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(tid), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('ex2_tokenization.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## 🌟 Exercise 3 — Understanding Token IDs and Special Prefixes

In [ ]:
# Print each token with its corresponding ID
print(f"{'Token':<20} {'ID':>6}")
print("-" * 28)
for tok, tid in zip(tokens, token_ids):
    prefix = " ← starts with Ġ (space before word)" if tok.startswith("Ġ") else ""
    print(f"  {tok:<18} {tid:>6}{prefix}")

### What does the special prefix `Ġ` indicate?

In GPT-2's BPE tokenizer, the character **`Ġ`** (Unicode U+0120) is used as a **space marker**. It indicates that there is a **whitespace before the token** in the original text — i.e., the token begins a new word.

For example:
- `"Artificial"` — no `Ġ` because it's the first token, there's no preceding space.
- `"Ġintelligence"` — the `Ġ` encodes the space between "Artificial" and "intelligence".
- `"Ġis"` — same: the space before "is" is encoded in the token itself.

This design choice means the tokenizer doesn't need a separate "space" token — the space information is **baked into the token** that follows it. This keeps the vocabulary efficient and prevents ambiguity when reconstructing the original text from token IDs.

In [ ]:
# Count tokens with and without the Ġ prefix
n_space    = sum(1 for t in tokens if t.startswith("Ġ"))
n_no_space = len(tokens) - n_space
print(f"Tokens with Ġ (word-internal space) : {n_space}")
print(f"Tokens without Ġ (start of text)    : {n_no_space}")

# Demonstrate: decode token IDs back to original text
decoded = tokenizer.decode(token_ids)
print(f"\nDecoded text : '{decoded}'")

## 🌟 Exercise 4 — Pretraining vs Fine-Tuning

### Pretraining

**Pretraining** is the first phase where a model is trained on a **massive, general-purpose corpus** of text (books, websites, Wikipedia, code, etc.) using a self-supervised objective. For GPT-style models the task is **causal language modeling**: given all previous tokens, predict the next token. For BERT-style models it's **masked language modeling**: predict randomly masked tokens.

No human-labeled data is needed — the supervision comes from the text itself. The model learns:
- Grammar and syntax
- World knowledge and facts
- Reasoning patterns
- Long-range dependencies in language

This phase is **extremely expensive** (weeks on thousands of GPUs), but it only needs to be done once. The result is a general-purpose model with rich representations of language.

---

### Fine-Tuning

**Fine-Tuning** is the second phase where the pretrained model is adapted to a **specific downstream task** using a **smaller, labeled dataset**. The model's weights — which already encode rich language understanding — are updated slightly (with a low learning rate) to specialize for the new task.

Examples:
- Fine-tune on customer support emails → specialized chatbot
- Fine-tune on medical records → clinical text classifier
- Fine-tune on instruction pairs (prompt → response) → instruction-following assistant (like ChatGPT via RLHF)

Fine-tuning is **cheap and fast** because the model already knows language; it just needs to learn the new task's format and domain.

---

| | Pretraining | Fine-Tuning |
|---|---|---|
| **Data** | Massive unlabeled corpus | Small labeled dataset |
| **Supervision** | Self-supervised (predict next/masked token) | Task-specific labels |
| **Cost** | Very high (days/weeks, thousands of GPUs) | Low (hours/days, few GPUs) |
| **Goal** | Learn general language representations | Adapt to a specific task |
| **Done by** | Large labs (OpenAI, Google, Meta) | Anyone with a task-specific dataset |

## 🌟 Exercise 5 — Generate Simple Text

In [ ]:
input_text = "The future of artificial intelligence will"

# Tokenize the input prompt
input_ids = tokenizer.encode(input_text, return_tensors="pt")

# Generate text: the model predicts the next token sequentially
# max_new_tokens=50 → generate 50 tokens beyond the prompt
# do_sample=True + temperature=0.8 → add controlled randomness (less repetitive)
# top_k=50 → sample only from the top-50 most probable tokens at each step
output_ids = model.generate(
    input_ids,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    pad_token_id=tokenizer.eos_token_id
)

output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Input  : {input_text}")
print(f"\nGenerated Output:")
print("-" * 60)
print(output_text)
print("-" * 60)
print(f"\nNew tokens generated: {output_ids.shape[1] - input_ids.shape[1]}")

In [ ]:
# Bonus: visualize next-token probabilities for the first generation step
model.eval()
with torch.no_grad():
    logits = model(input_ids).logits  # (1, seq_len, vocab_size)

next_token_logits = logits[0, -1, :]               # logits for the very next token
probs             = torch.softmax(next_token_logits, dim=-1)
top_k_probs, top_k_ids = torch.topk(probs, 15)

top_k_tokens = [tokenizer.decode([tid.item()]).strip() for tid in top_k_ids]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_k_tokens[::-1], top_k_probs.numpy()[::-1], color='#4e91d4')
ax.set_xlabel('Probability')
ax.set_title(f'Top-15 Next Token Probabilities\nPrompt: "{input_text}"', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
for bar, p in zip(bars, top_k_probs.numpy()[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{p:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ex5_next_token_probs.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved ✓")

## Summary & Key Takeaways

| Exercise | Concept | Key point |
|---|---|---|
| 1 | LLMs | Transformer-based models trained on massive text corpora for NLP tasks |
| 2 | Tokenization | BPE splits text into sub-word tokens, each mapped to a unique integer ID |
| 3 | Special prefix `Ġ` | Encodes a leading space — word boundary marker in GPT-2's BPE scheme |
| 4 | Pretraining vs Fine-Tuning | Pretraining = expensive + general; Fine-Tuning = cheap + task-specific |
| 5 | Text generation | GPT-2 samples from a probability distribution over vocabulary at each step |

**How GPT-2 generates text:**
1. Tokenize the prompt → integer IDs
2. Pass IDs through the Transformer → logits over the full vocabulary
3. Apply softmax → probability distribution over next tokens
4. Sample (or argmax) a token → append to sequence
5. Repeat until `max_new_tokens` is reached or `<|endoftext|>` is generated

Each step is **autoregressive** — the model conditions on all previously generated tokens.